In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:

import os
import random
import matplotlib.pyplot as plt
import sys

import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from CTC_tools import (load_selected_embeddings_ctc,
                       CTCPhonemeDataset,
                       ctc_collate_fn,
                       train_ctc,
                       ctc_greedy_decode,
                       CTCModel_lstm,
                       CTCModel_linear)

sys.path.append(r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project')

from evaluation_tools import cer_detailed
#from VizualisationTools import cer_detailed_viz
import panphon.distance
from ToIPA import corpres2ipa
from tqdm import tqdm



In [ ]:
root_dir = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
int_code = [random.randint(65, 90) for i in range(10)]
uid = ''.join([chr(code) for code in int_code])
save_path = str(CTC_MODELS_DIR)
result_dir = os.path.join(save_path, uid)
# sequences, targets, input_lengths, info — из нашей функции load_embeddings_for_ctc
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']
# phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil']
phoneme_list = phoneme_list_full + ['sil']
# phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l',
#                 "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'",
#                 'y0', 'z', "z'", 'zh', 'ch' 'sil', 'a1', 'i1','u1', 'y1', 'y4', 'u4', ]

train_speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna']
test_speakers = ['Galina', 'Victoria', 'Petr', 'Alexander']



In [ ]:
train_sequences, train_targets, train_info = load_selected_embeddings_ctc(
    root_dir=root_dir,
    phoneme_list=phoneme_list,
    speakers=train_speakers,
    max_sequences_per_speaker=1000)

test_sequences, test_targets, test_info = load_selected_embeddings_ctc(
    root_dir=root_dir,
    phoneme_list=phoneme_list,
    speakers=test_speakers,
    max_sequences_per_speaker=1000)

In [ ]:
train_dataset = CTCPhonemeDataset(
    train_sequences,
    train_targets,
    phoneme_list
)

test_dataset = CTCPhonemeDataset(
    test_sequences,
    test_targets,
    phoneme_list
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=ctc_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=ctc_collate_fn
)

In [ ]:
x, y, x_lens, y_lens = next(iter(train_loader))
print(x.shape)        # [B, T, D]
print(y.shape)        # [sum(target_lengths)]
print(x_lens)
print(y_lens)

In [ ]:
input_dim = train_sequences[0].shape[1]
hidden_dim = 256
num_phonemes = len(phoneme_list)

model = CTCModel_lstm(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_phonemes=num_phonemes
)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
num_epochs = 20

writer = SummaryWriter(log_dir=result_dir)
best_model_path = os.path.join(result_dir, 'best_model.pth')

train_ctc(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=num_epochs,
    device=device,
    writer=writer,
    save_path=best_model_path
)
writer.close()

In [ ]:
best_model_path

In [ ]:
model.eval()
with torch.no_grad():
    x, y, x_lens, y_lens = next(iter(test_loader))
    x = x.to(device)
    print(x_lens)
    logits = model(x)
    preds = ctc_greedy_decode(
        logits,
        x_lens,
        train_dataset.idx2phoneme
    )

    print("PRED:", preds[0])

In [ ]:
def unpack_targets(targets, target_lengths, idx2phoneme):
    true_phonemes = []

    offset = 0
    for L in target_lengths:
        seq = targets[offset:offset + L]
        offset += L

        true_phonemes.append(
            [idx2phoneme[int(i)] for i in seq]
        )

    return true_phonemes

In [ ]:
model.eval()

cers = []

with torch.no_grad():
    for x, targets, x_lens, y_lens in test_loader:
        x = x.to(device)

        logits = model(x)  # [B, T, C]

        # PRED
        pred_phonemes = ctc_greedy_decode(
            logits,
            x_lens,
            test_dataset.idx2phoneme
        )

        # TRUE
        true_phonemes = unpack_targets(
            targets,
            y_lens,
            test_dataset.idx2phoneme
        )

        # CER
        for true_seq, pred_seq in zip(true_phonemes, pred_phonemes):
            cer_res, subs, ins, del_ = cer_detailed(true_seq, pred_seq)
            cers.append(cer_res)

        break  # сначала смотри один батч

In [ ]:
for i in range(min(5, len(pred_phonemes))):
    print(f"\nSample {i}")
    print("TRUE:", " ".join(true_phonemes[i]))
    print("PRED:", " ".join(pred_phonemes[i]))
    print("CER :", cer_detailed(true_phonemes[i], pred_phonemes[i]))
    print("Substitutions:", subs)
    print("Insertions:", ins)
    print("Deletions:", del_)

In [ ]:
model.eval()

cers = []
subs_count = 0
ins_count = 0
del_count = 0

subs_list = []
ins_list = []
dels_list = []

subs_count_avg = 0
ins_count_avg = 0
del_count_avg = 0

dst = panphon.distance.Distance()
overall_pfer = []

with torch.no_grad():
    for x, targets, x_lens, y_lens in test_loader:
        x = x.to(device)

        logits = model(x)  # [B, T, C]

        # PRED
        pred_phonemes = ctc_greedy_decode(
            logits,
            x_lens,
            test_dataset.idx2phoneme
        )

        # TRUE
        true_phonemes = unpack_targets(
            targets,
            y_lens,
            test_dataset.idx2phoneme
        )

        # CER
        for true_seq, pred_seq in zip(true_phonemes, pred_phonemes):

            cer_res, subs, ins, del_ = cer_detailed(true_seq, pred_seq)
            cers.append(cer_res)
            subs_count += subs
            ins_count += ins
            del_count += del_
            subs_list.append(subs)
            ins_list.append(ins)
            dels_list.append(del_)
            subs_count_avg += subs/len(true_seq)
            ins_count_avg += ins/len(true_seq)
            del_count_avg += del_/len(true_seq)
            
            ipa_pred = corpres2ipa(pred_seq)
            ipa_true = corpres2ipa(true_seq)
            true_len = len(true_seq)
            dist = dst.feature_edit_distance(ipa_pred, ipa_true)
            
            overall_pfer.append(dist/true_len)

mean_cer = sum(cers) / len(cers)
print(f"\nMean CER: {mean_cer:.4f}")
print("Substitutions:", subs_count, subs_count_avg, subs_count_avg / len(cers))
print("Insertions:", ins_count, ins_count_avg, ins_count_avg / len(cers))
print("Deletions:", del_count, del_count_avg, del_count_avg / len(cers))

In [ ]:
sum(overall_pfer)/len(overall_pfer)

In [ ]:
#result = cer_detailed_viz(cers,subs_list, ins_list, dels_list)

In [ ]:
plt.plot(ins_list, range(len(subs_list)))

In [ ]:
plt.plot(dels_list, range(len(subs_list)))

In [ ]:
len(cers)

In [ ]:
def get_stresses(phonemes):
    stresses = []
    for phoneme in phonemes:
        print(phoneme)
        if len(phoneme) == 2:
            if phoneme[1] in '124':
                stresses.append('-')
            elif phoneme[1] == '0':
                    stresses.append('+')

    return stresses


In [ ]:
model.eval()

cers = []
stresses_rate = 0

with torch.no_grad():
    for x, targets, x_lens, y_lens in test_loader:
        x = x.to(device)

        logits = model(x)  # [B, T, C]

        # PRED
        pred_phonemes = ctc_greedy_decode(
            logits,
            x_lens,
            test_dataset.idx2phoneme
        )

        # TRUE
        true_phonemes = unpack_targets(
            targets,
            y_lens,
            test_dataset.idx2phoneme
        )

        # CER
        for true_seq, pred_seq in zip(true_phonemes, pred_phonemes):
            cers.append(cer_detailed(true_seq, pred_seq))

mean_cer = sum(cers) / len(cers)
print(f"\nMean CER: {mean_cer:.4f}")


In [ ]:
mean_stresses = stresses_rate / len(cers)
print(f"\nstresses rate: {mean_stresses:.4f}")

In [ ]:
model.eval()

stresses_rate = 0

with torch.no_grad():
    for x, targets, x_lens, y_lens in test_loader:
        x = x.to(device)

        logits = model(x)  # [B, T, C]

        pred_phonemes = ctc_greedy_decode(
            logits,
            x_lens,
            test_dataset.idx2phoneme
        )

        true_phonemes = unpack_targets(
            targets,
            y_lens,
            test_dataset.idx2phoneme
        )

        # CER
        for true_seq, pred_seq in zip(true_phonemes, pred_phonemes):

            true_stresses = get_stresses(true_seq)
            pred_stresses = get_stresses(pred_seq)
            if true_stresses == pred_stresses:
                stresses_rate += 1

                
mean_stresses = stresses_rate / len(cers)
print(f"\nstresses rate: {mean_stresses:.4f}")


In [ ]:
true_stresses

In [ ]:
pred_stresses

In [ ]:
''.join(true_seq)

In [ ]:
''.join(pred_seq)

In [ ]:
for i,j in zip(true_seq, pred_seq):
    print(i, j)

In [ ]:
best_model_path